# Phase 14 — 5-Fold Cross-Validation για Hyperparameter Tuning

Χρησιμοποιούμε 5-fold CV για να βρούμε τον βέλτιστο συνδυασμό παραμέτρων.

**Παράμετροι που δοκιμάζουμε:**
- `LR`: 1e-5, 2e-5, 3e-5
- `MAX_LENGTH`: 128, 256

**Μοντέλο:** DistilBERT train+valid, σταθερά 20 epochs


In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Συνδυασμός train + valid
train_full = pd.concat([train, valid], ignore_index=True).reset_index(drop=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)

# Label Encoding
le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_hazard  = le_hazard.transform(train_full['hazard-category'])
y_product = le_product.transform(train_full['product-category'])

# Συνδυαστικό label για stratified split (hazard + product)
y_combined = [f'{h}_{p}' for h, p in zip(train_full['hazard-category'], train_full['product-category'])]

print(f'Train+Valid: {len(train_full)} δείγματα')
print(f'Hazard classes:  {len(le_hazard.classes_)}')
print(f'Product classes: {len(le_product.classes_)}')

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
N_EPOCHS   = 20
BATCH_SIZE = 32
N_FOLDS    = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')


class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    return (f1_hazard + f1_product) / 2

In [ ]:
def run_cv(lr, max_length, n_epochs=N_EPOCHS):
    """
    Τρέχει 5-fold CV για δοσμένα hyperparameters.
    Επιστρέφει το μέσο ST1 score.
    """
    print(f'\n{"="*50}')
    print(f'LR={lr}, MAX_LENGTH={max_length}, EPOCHS={n_epochs}')
    print(f'{"="*50}')

    skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []
    texts  = np.array(texts_full)

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, y_combined)):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')

        # Split
        fold_texts_train = texts[train_idx].tolist()
        fold_texts_val   = texts[val_idx].tolist()
        fold_y_hazard_train  = y_hazard[train_idx]
        fold_y_hazard_val    = y_hazard[val_idx]
        fold_y_product_train = y_product[train_idx]
        fold_y_product_val   = y_product[val_idx]

        # DataLoaders
        train_loader_h = DataLoader(FoodHazardDataset(fold_texts_train, fold_y_hazard_train,  tokenizer, max_length), batch_size=BATCH_SIZE, shuffle=True)
        train_loader_p = DataLoader(FoodHazardDataset(fold_texts_train, fold_y_product_train, tokenizer, max_length), batch_size=BATCH_SIZE, shuffle=True)
        val_loader_h   = DataLoader(FoodHazardDataset(fold_texts_val,   fold_y_hazard_val,    tokenizer, max_length), batch_size=BATCH_SIZE, shuffle=False)
        val_loader_p   = DataLoader(FoodHazardDataset(fold_texts_val,   fold_y_product_val,   tokenizer, max_length), batch_size=BATCH_SIZE, shuffle=False)

        # Hazard model
        model_h = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le_hazard.classes_)).to(device)
        opt_h   = AdamW(model_h.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_loader_h) * n_epochs
        sch_h = get_linear_schedule_with_warmup(opt_h, int(total_steps*0.1), total_steps)

        for epoch in range(n_epochs):
            loss = train_epoch(model_h, train_loader_h, opt_h, sch_h)
            print(f'  Hazard Epoch {epoch+1}/{n_epochs} | Loss: {loss:.4f}', end='\r')

        # Product model
        model_p = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le_product.classes_)).to(device)
        opt_p   = AdamW(model_p.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_loader_p) * n_epochs
        sch_p = get_linear_schedule_with_warmup(opt_p, int(total_steps*0.1), total_steps)

        for epoch in range(n_epochs):
            loss = train_epoch(model_p, train_loader_p, opt_p, sch_p)
            print(f'  Product Epoch {epoch+1}/{n_epochs} | Loss: {loss:.4f}', end='\r')

        # Αξιολόγηση
        preds_h = le_hazard.inverse_transform(get_predictions(model_h, val_loader_h))
        preds_p = le_product.inverse_transform(get_predictions(model_p, val_loader_p))
        true_h  = le_hazard.inverse_transform(fold_y_hazard_val)
        true_p  = le_product.inverse_transform(fold_y_product_val)

        score = official_st1_score(true_h, preds_h, true_p, preds_p)
        scores.append(score)
        print(f'  Fold {fold+1} ST1 Score: {score:.4f}')

        # Καθαρισμός μνήμης GPU
        del model_h, model_p
        torch.cuda.empty_cache()

    mean_score = np.mean(scores)
    std_score  = np.std(scores)
    print(f'\n→ CV Score: {mean_score:.4f} ± {std_score:.4f}')
    print(f'  Fold scores: {[f"{s:.4f}" for s in scores]}')
    return mean_score, std_score

## Εκτέλεση CV


In [ ]:
# Baseline — ίδιο με best submission
score_baseline, std_baseline = run_cv(lr=2e-5, max_length=128)

In [ ]:
# LR μικρότερο
score_lr1, std_lr1 = run_cv(lr=1e-5, max_length=128)

In [ ]:
# LR μεγαλύτερο
score_lr3, std_lr3 = run_cv(lr=3e-5, max_length=128)

In [ ]:
# MAX_LENGTH μεγαλύτερο
score_len256, std_len256 = run_cv(lr=2e-5, max_length=256)

In [ ]:
# Σύνοψη αποτελεσμάτων
print('\n' + '='*50)
print('ΣΥΝΟΨΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ CV')
print('='*50)
results = [
    ('LR=2e-5, MAX_LEN=128 (baseline)', score_baseline, std_baseline),
    ('LR=1e-5, MAX_LEN=128',            score_lr1,      std_lr1),
    ('LR=3e-5, MAX_LEN=128',            score_lr3,      std_lr3),
    ('LR=2e-5, MAX_LEN=256',            score_len256,   std_len256),
]
results.sort(key=lambda x: x[1], reverse=True)
for name, score, std in results:
    print(f'{name:40s} → {score:.4f} ± {std:.4f}')

best_name, best_score, _ = results[0]
print(f'\nΒέλτιστο: {best_name} με CV score {best_score:.4f}')